# Plant NeRF — MASt3R poses (train, validate, export, save)

Trains `nerfacto` on the images using **MASt3R** poses, validates the model,
exports multiple output formats (point cloud / mesh / video), and saves everything
to Drive.

**Before running:** upload `nerf_ready_dataset.zip` (built by the terminal command) to your
Drive, e.g. `MyDrive/plant_pheno/nerf_ready_dataset.zip`, and set `ZIP_PATH` below.


In [1]:
# CELL 1 — Mount Drive, set paths
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

import os

ZIP_PATH   = "/content/drive/MyDrive/jetcobot2026/nerf_ready_dataset.zip"  # <-- edit if needed
WORK       = "/content/work"
DRIVE_OUT  = "/content/drive/MyDrive/jetcobot2026/output"      # everything final gets copied here

os.makedirs(WORK, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)
print("Zip source:", ZIP_PATH)
print("Drive output folder:", DRIVE_OUT)


Zip source: /content/drive/MyDrive/jetcobot2026/nerf_ready_dataset.zip
Drive output folder: /content/drive/MyDrive/jetcobot2026/output


In [2]:
# CELL 2 — Install nerfstudio + deps (GPU runtime required: Runtime > Change runtime type > GPU)
!pip install -q --upgrade pip
!pip install -q nerfstudio open3d trimesh
!ns-install-cli || true
print("nerfstudio install done")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-language 2.21.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-spanner 3.68.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
google-api-core 2.30.3 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
proto-plus 1.28.1

In [4]:
# CELL 2b — Patch torch.load for newer PyTorch (2.6+) where weights_only defaults to True.
# nerfstudio checkpoints/configs are full Python objects, not just tensors, so ns-eval/ns-export/ns-render
# fail with an UnpicklingError unless weights_only=False. Writing a sitecustomize.py into site-packages
# makes the patch apply automatically inside every subprocess (the !ns-train / !ns-eval / !ns-export calls
# below run as separate processes, so a patch only in this notebook's Python session wouldn't reach them).
import site, os

sitepkg = site.getsitepackages()[0]
sitecustomize_path = os.path.join(sitepkg, "sitecustomize.py")
with open(sitecustomize_path, "w") as f:
    f.write(
        "import torch\n"
        "_orig_load = torch.load\n"
        "def _patched_load(*args, **kwargs):\n"
        "    kwargs.setdefault('weights_only', False)\n"
        "    return _orig_load(*args, **kwargs)\n"
        "torch.load = _patched_load\n"
    )
print("Patched torch.load (weights_only=False) via", sitecustomize_path)


Patched torch.load (weights_only=False) via /usr/local/lib/python3.12/dist-packages/sitecustomize.py


In [5]:
# CELL 3 — Unzip the combined dataset
import zipfile, shutil, glob

extract_dir = os.path.join(WORK, "raw")
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(extract_dir)

# handle the case where zip contains a top-level 'nerf_ready_dataset' folder
root_candidates = glob.glob(os.path.join(extract_dir, '*'))
DATA_ROOT = extract_dir if os.path.exists(os.path.join(extract_dir, 'images')) else root_candidates[0]
print("DATA_ROOT:", DATA_ROOT)
print(os.listdir(DATA_ROOT))


DATA_ROOT: /content/work/raw/nerf_ready_dataset
['images', 'colmap', 'mast3r']


In [6]:
# CELL 4 — Build the MASt3R-based nerfstudio dataset (transforms.json, camera-to-world, OpenGL convention)
import numpy as np, json

mast3r_ds = os.path.join(WORK, "dataset_mast3r")
os.makedirs(os.path.join(mast3r_ds, "images"), exist_ok=True)
shutil.copytree(os.path.join(DATA_ROOT, "images"), os.path.join(mast3r_ds, "images"), dirs_exist_ok=True)

poses_c2w  = np.load(os.path.join(DATA_ROOT, "mast3r", "poses_c2w.npy"))   # (N,4,4) camera-to-world, OpenCV axes
intrinsics = np.load(os.path.join(DATA_ROOT, "mast3r", "intrinsics.npy"))  # (N,3,3) or (3,3)

img_files = sorted(
    [f for f in os.listdir(os.path.join(mast3r_ds, "images")) if f.lower().endswith(('.png', '.jpg', '.jpeg'))],
    key=lambda x: int(''.join(filter(str.isdigit, x)) or 0)
)
assert len(img_files) == poses_c2w.shape[0], f"images ({len(img_files)}) != poses ({poses_c2w.shape[0]})"

from PIL import Image
w, h = Image.open(os.path.join(mast3r_ds, "images", img_files[0])).size

# OpenCV -> OpenGL/NeRF axis flip: negate Y and Z columns of rotation
cv_to_gl = np.diag([1, -1, -1, 1]).astype(np.float64)

def get_K(i):
    return intrinsics[i] if intrinsics.ndim == 3 else intrinsics

frames = []
for i, fname in enumerate(img_files):
    pose = poses_c2w[i].astype(np.float64) @ cv_to_gl
    K = get_K(i)
    frames.append({
        "file_path": f"images/{fname}",
        "transform_matrix": pose.tolist(),
        "fl_x": float(K[0, 0]), "fl_y": float(K[1, 1]),
        "cx": float(K[0, 2]),   "cy": float(K[1, 2]),
        "w": w, "h": h,
    })

transforms = {
    "camera_model": "OPENCV",
    "w": w, "h": h,
    "fl_x": frames[0]["fl_x"], "fl_y": frames[0]["fl_y"],
    "cx": frames[0]["cx"], "cy": frames[0]["cy"],
    "frames": frames,
}

with open(os.path.join(mast3r_ds, "transforms.json"), "w") as f:
    json.dump(transforms, f, indent=2)

print("MASt3R dataset ready at:", mast3r_ds, "|", len(frames), "frames")


MASt3R dataset ready at: /content/work/dataset_mast3r | 139 frames


In [7]:
# CELL 5 — Train NeRF: MASt3R poses
!ns-train nerfacto \
  --data {mast3r_ds}/transforms.json \
  --output-dir {WORK}/outputs \
  --experiment-name mast3r_run \
  --viewer.quit-on-train-completion True \
  --max-num-iterations 15000 \
  nerfstudio-data


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
11400 (76.00%)      300.004 ms           18 m, 0 s            14.32 K                                
11410 (76.07%)      295.332 ms           17 m, 40 s           14.54 K                                
11420 (76.13%)      309.448 ms           18 m, 27 s           14.02 K                                
11430 (76.20%)      300.256 ms           17 m, 51 s           14.29 K                                
11440 (76.27%)      300.868 ms           17 m, 51 s           14.27 K                                
---------------------------------------------------------------------------------------------------- 
Viewer running locally at: http://localhost:7007 (listening on 0.0.0.0)                              
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec                       
-----------------------------------------------------------------------------------                  
1136

In [8]:
# CELL 6 — Locate the trained config
import glob

def latest_config(exp_name):
    matches = sorted(glob.glob(f"{WORK}/outputs/{exp_name}/nerfacto/*/config.yml"))
    assert matches, f"no config found for {exp_name}"
    return matches[-1]

mast3r_cfg = latest_config("mast3r_run")
print("MASt3R config:", mast3r_cfg)


MASt3R config: /content/work/outputs/mast3r_run/nerfacto/2026-07-24_154226/config.yml


In [10]:
!ls -lh /content/work/outputs/mast3r_run/nerfacto/2026-07-24_154226/nerfstudio_models

total 232M
-rw-r--r-- 1 root root 232M Jul 24 17:01 step-000014999.ckpt


In [18]:
from pathlib import Path

eval_utils = Path("/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py")

text = eval_utils.read_text()

old = 'loaded_state = torch.load(load_path, map_location="cpu")'
new = 'loaded_state = torch.load(load_path, map_location="cpu", weights_only=False)'

if old in text:
    text = text.replace(old, new)
    eval_utils.write_text(text)
    print("✅ Patched eval_utils.py successfully.")
else:
    print("⚠️ Expected line not found. Searching for torch.load...")

    for i, line in enumerate(text.splitlines(), 1):
        if "torch.load" in line:
            print(f"Line {i}: {line}")

✅ Patched eval_utils.py successfully.


In [19]:
!ns-eval \
  --load-config /content/work/outputs/mast3r_run/nerfacto/2026-07-24_154226/config.yml \
  --output-path /content/work/eval/mast3r_metrics.json \
  2>&1 | tee /content/work/eval/ns_eval.log

2026-07-24 17:11:11.798598: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[17:11:18] Auto image downscale factor of 1                                                 nerfstudio_dataparser.py:484
2026-07-24 17:11:20.787491: I tensorflow/core/platfo

In [20]:
# CELL 8 — Export multiple output types (point cloud, poisson mesh, TSDF mesh) + turntable video
export_dir = os.path.join(WORK, "exports")

# pointcloud/poisson need surface normals; --normal-method open3d computes them directly from the
# point geometry, so it works even though nerfacto wasn't trained with --pipeline.model.predict-normals.
# This is the automated equivalent of computing normals in Open3D by hand after the fact.
normal_args = {"pointcloud": "--normal-method open3d", "poisson": "--normal-method open3d", "tsdf": ""}

for out_type in ["pointcloud", "poisson", "tsdf"]:
    out_path = os.path.join(export_dir, "mast3r", out_type)
    os.makedirs(out_path, exist_ok=True)
    extra = normal_args[out_type]
    print(f"--- exporting {out_type} for mast3r ---")
    !ns-export {out_type} --load-config {mast3r_cfg} --output-dir {out_path} {extra}

# also render a turntable flythrough video, useful for visual comparison
render_path = os.path.join(export_dir, "mast3r", "render.mp4")
!ns-render spiral --load-config {mast3r_cfg} --output-path {render_path} || echo "render skipped, define a camera path if this fails"


--- exporting pointcloud for mast3r ---
2026-07-24 17:15:13.418299: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[17:15:21] Auto image downscale factor of 1                                                 ]8;id=729136;file:///usr/local/lib/pytho

In [ ]:
render_path = os.path.join(export_dir, "mast3r", "render.mp4")
!ns-render spiral \
  --load-config {mast3r_cfg} \
  --output-path {render_path} \
  --render-nearest-camera True \
  || echo "render skipped, define a camera path if this fails"

2026-07-24 17:57:03.767345: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[17:57:09] Auto image downscale factor of 1                                                 ]8;id=5559;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/data/datapa

In [22]:
import os
import shutil

# Destination in Google Drive
DEST = "/content/drive/MyDrive/work_backup"

if not os.path.exists(DEST):
    shutil.copytree(WORK, DEST)
    print(f"✅ Backup created:\n{WORK}\n➡️ {DEST}")
else:
    print(f"ℹ️ Backup already exists, skipping copy:\n{DEST}")

✅ Copied:
/content/work

➡️ /content/drive/MyDrive/work_backup


In [ ]:
# CELL 9 — Save EVERYTHING to Drive (config, metrics, exports, render, checkpoint)
import pandas as pd

final_dir = os.path.join(DRIVE_OUT, "run_" + pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
os.makedirs(final_dir, exist_ok=True)

shutil.copytree(eval_dir, os.path.join(final_dir, "evaluation"))
shutil.copytree(os.path.join(export_dir, "mast3r"), os.path.join(final_dir, "exports"))

# also copy the full model checkpoint/config for later reuse in nerfstudio
shutil.copytree(os.path.dirname(mast3r_cfg), os.path.join(final_dir, "checkpoint"))

print("MASt3R RESULTS SAVED TO:", final_dir)
print("Copy this path into the comparison notebook as MAST3R_RESULTS_DIR.")
!ls -R {final_dir} | head -60


## Notes
- **Re-opening later in nerfstudio**: point `ns-viewer --load-config <checkpoint/config.yml>` at the saved checkpoint folder to interactively re-view the trained model.
- **CloudCompare / MeshLab / Blender**: open any `.ply` under `exports/pointcloud|poisson|tsdf/`.
- `--max-num-iterations 30000` is a reasonable default; drop to 10000–15000 first if you just want a fast sanity check before committing to a full run.
- The printed `final_dir` path above is what you paste into the comparison notebook.
